# Capítulo 3 — Aprendizado federado: clientes maliciosos e privacidade

## Missão
A Fortaleza Digital quer treinar um modelo com dados de 10 filiais **sem juntar os dados** (privacidade, LGPD). A técnica é o **aprendizado federado**: cada filial treina localmente e envia só os pesos. Mas e se uma filial for maliciosa? Você vai:
- escrever o **FedAvg** do zero;
- sofrer um ataque de **envenenamento de modelo**;
- defender com **agregação robusta** (mediana e média aparada) e com detecção de atualizações anômalas, como o Auror do cap. 4;
- testar um ataque de **inferência de pertencimento**, que descobre se um exemplo estava no treino.

**No fim deste caderno você terá:** o módulo `meu_escudo/federado.py` (treino local em NumPy, FedAvg, mediana, média aparada e detector de clientes anômalos) e o ataque de inferência de pertencimento.

**Leitura essencial (opcional, livro em inglês):**
- 4.2 Background (collaborative deep learning) · PDF p. 79
- 4.3 Auror: An Automated Defense · PDF p. 83
- 4.4 A New CDL Attack: GAN Attack · PDF p. 85
- 12.2 Data Security and Federated Learning · PDF p. 171
- 12.4 Challenges · PDF p. 175

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap03_aprendizado_federado_clientes_maliciosos_e_pr.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_escudo/
os.makedirs("../meu_escudo", exist_ok=True)
open("../meu_escudo/__init__.py", "a").close()        # meu_escudo/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_escudo")

## Ideia em 30 segundos
**FedAvg**, a cada rodada:
1. o servidor manda os pesos globais;
2. cada cliente treina algumas épocas nos próprios dados;
3. o servidor faz a **média** dos pesos recebidos.

Um cliente malicioso pode mandar pesos absurdos e arrastar a média. A **mediana** coordenada a coordenada e a **média aparada** (descarta os extremos) aguentam até cerca de metade de clientes ruins.

### 3.1 **O federado.** Crie `federado.py` com:
- `treino_local(W, X, y, epocas=5, taxa=0.5)`: regressão softmax por gradiente (W de formato 65 × 10, com viés);
- `prever(W, X)`;
- `agregar(pesos, metodo="media" | "mediana" | "aparada", corte=0.2)`;
- `rodar(clientes, rodadas, metodo, maliciosos=(), forca=10)`: os clientes maliciosos mandam `W_global - forca·(W_local - W_global)`, a direção invertida e amplificada. Devolve a acurácia por rodada num teste;
- `suspeitos(pesos, k=2.5)`: os clientes cuja distância até a mediana passa de `k ×` a mediana das distâncias.

_A célula grava `meu_escudo/federado.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_escudo/federado.py


### 3.2 **Dez filiais.** Separe 25% dos dígitos para o teste e divida o resto em 10 clientes de tamanhos parecidos (`np.array_split` de uma permutação). Rode 15 rodadas de FedAvg sem ataque e compare a acurácia final com a de um modelo centralizado (`LogisticRegression` com todos os dados de treino).

### 3.3 **O ataque e as defesas.** Com 2 clientes maliciosos (índices 3 e 7), rode 15 rodadas com média, mediana e média aparada (corte 0,2). Desenhe a acurácia por rodada das três agregações.

### 3.4 **Detectar quem trapaceia.** Numa rodada, colete os 10 pesos locais (com os clientes 3 e 7 maliciosos) e use `suspeitos`. Ele acha exatamente os maliciosos? Com 4 maliciosos (1, 3, 5 e 7), ainda acha?

### 3.5 **Inferência de pertencimento.** Um modelo sobreajustado fica **mais confiante** nos exemplos que viu no treino, e isso vaza privacidade.
1. Treine uma `RandomForestClassifier(200)` em só 300 exemplos, com folhas de 1 exemplo (decora o treino).
2. Como atacante, use a confiança na classe **certa** como escore de "estava no treino".
3. Calcule a AUC do ataque: 300 membros contra 300 não membros.

Repita com `min_samples_leaf=10` (menos sobreajuste): o vazamento diminui? E a acurácia?

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
